In [1]:
import sys
import os

# ensure we're always running from project root
os.chdir(os.path.dirname(os.getcwd()))
sys.path.insert(0, '.')

In [2]:
from src.load import load_data
from src.clean import clean_data
import pandas as pd

df_raw = load_data()
df = clean_data(df_raw)

── Data loaded ──────────────────────────
Rows:              1,033,019
Date range:        2009-12-01 → 2011-12-09
Unique customers:  5,940
Cancellations:     19,100
Null Customer IDs: 235,150
─────────────────────────────────────────
── Cleaning summary ─────────────────────
Rows before: 1,033,019
Rows after:  1,021,374
Removed:     11,645
Outlier rows flagged: 7
─────────────────────────────────────────


cancelled quantity / total ordered quantity per product

# Cancelled quantity

In [6]:
canc_df = df[df['Invoice'].str.startswith('C')]
canc_qnt = canc_df.groupby(['Description'])['Quantity'].sum().abs()
canc_qnt

Description
 3 STRIPEY MICE FELTCRAFT             2
 50'S CHRISTMAS GIFT BAG LARGE        2
 CHERRY BLOSSOM  DECORATIVE FLASK     3
 DOLLY GIRL BEAKER                    7
 FLAMINGO LIGHTS                      2
                                     ..
ZINC SWEETHEART WIRE LETTER RACK      4
ZINC T-LIGHT HOLDER STAR LARGE       11
ZINC T-LIGHT HOLDER STARS SMALL      44
ZINC TOP  2 DOOR WOODEN SHELF        11
ZINC WILLIE WINKIE  CANDLE STICK     12
Name: Quantity, Length: 3065, dtype: int64

# Sales Quantity

In [7]:
sales_df = df[~df['Invoice'].str.startswith('C')]
sales_qnt = sales_df.groupby(['Description'])['Quantity'].sum()
sales_qnt

Description
  DOORMAT UNION JACK GUNS AND ROSES     177
 3 STRIPEY MICE FELTCRAFT               691
 4 PURPLE FLOCK DINNER CANDLES          335
 50'S CHRISTMAS GIFT BAG LARGE         1915
 ANIMAL STICKERS                        385
                                       ... 
ZINC T-LIGHT HOLDER STARS SMALL        5089
ZINC TOP  2 DOOR WOODEN SHELF           249
ZINC WILLIE WINKIE  CANDLE STICK       6794
ZINC WIRE KITCHEN ORGANISER              30
ZINC WIRE SWEETHEART LETTER TRAY         83
Name: Quantity, Length: 5387, dtype: int64

# Merge

In [8]:
product_df = sales_qnt.reset_index().merge(
    canc_qnt.reset_index(),
    on='Description',
    how='left'
)
product_df.columns = ['Description', 'Ordered', 'Cancelled']
product_df['Cancelled'] = product_df['Cancelled'].fillna(0)
product_df['Cancellation Rate'] = (product_df['Cancelled'] / product_df['Ordered'] * 100).round(2)

In [9]:
product_df

,Description,Ordered,Cancelled,Cancellation Rate
0,DOORMAT UNION JACK GUNS AND ROSES,177,0.0,0.00
1,3 STRIPEY MICE FELTCRAFT,691,2.0,0.29
2,4 PURPLE FLOCK DINNER CANDLES,335,0.0,0.00
3,50'S CHRISTMAS GIFT BAG LARGE,1915,2.0,0.10
4,ANIMAL STICKERS,385,0.0,0.00
...,...,...,...,...
5382,ZINC T-LIGHT HOLDER STARS SMALL,5089,44.0,0.86
5383,ZINC TOP 2 DOOR WOODEN SHELF,249,11.0,4.42
5384,ZINC WILLIE WINKIE CANDLE STICK,6794,12.0,0.18
5385,ZINC WIRE KITCHEN ORGANISER,30,0.0,0.00


In [15]:
def calc_product_cancellation_rates(df: pd.DataFrame, min_quantity_percentile: float = 0.25) -> pd.DataFrame:
    canc_df = df[df['Invoice'].str.startswith('C', na=False)]
    canc_qnt = canc_df.groupby(['Description'])['Quantity'].sum().abs()

    sales_df = df[~df['Invoice'].str.startswith('C', na=False)]
    sales_qnt = sales_df.groupby(['Description'])['Quantity'].sum()

    product_df = sales_qnt.reset_index().merge(
        canc_qnt.reset_index(),
        on='Description',
        how='left'
    )
    product_df.columns = ['Description', 'Ordered', 'Cancelled']
    product_df['Cancelled'] = product_df['Cancelled'].fillna(0)
    product_df['Cancellation Rate'] = (product_df['Cancelled'] / product_df['Ordered'] * 100).round(2)

    threshold = product_df['Ordered'].quantile(min_quantity_percentile)

    product_df = product_df[product_df['Ordered'] >= threshold]

    product_df = product_df[product_df['Cancellation Rate'] < 100]
    return product_df.sort_values('Cancellation Rate', ascending=False).reset_index(drop=True)

In [16]:
calc_product_cancellation_rates(df).head(10)

,Description,Ordered,Cancelled,Cancellation Rate
0,RED POLKADOT PUDDING BOWL,3717,3648.0,98.14
1,BLUE POLKADOT PUDDING BOWL,2565,2496.0,97.31
2,MULTICOLOUR POLKADOT PLATE,713,684.0,95.93
3,MEDIUM CERAMIC TOP STORAGE JAR,78033,74494.0,95.46
4,ROSES ON BLUE TEACUP CANDLE,539,504.0,93.51
5,SET OF 6 DOTS CHOPSTICKS,211,181.0,85.78
6,LA PALMIERA WALL THERMOMETER,452,372.0,82.30
7,PANTRY CHOPPING BOARD,1154,946.0,81.98
8,BLUE POLKADOT PURSE,415,330.0,79.52
9,PORCELAIN CHERUB BELL SMALL,329,260.0,79.03
